In [1]:
import numpy as np
import pandas as pd 

In [2]:
bioinfo = pd.read_parquet('../data/raw/bioinfo.parquet')
deology = pd.read_parquet('../data/raw/ideology.parquet')
terms = pd.read_parquet('../data/raw/terms.parquet')

bill_summaries_119 = pd.read_parquet('../data/raw/bill_summaries_119.parquet')
sponsored_legislation = pd.read_parquet('../data/raw/sponsored_legislation.parquet')
committee_members = pd.read_parquet('../data/raw/committee_members.parquet')
committees = pd.read_parquet('../data/raw/committees.parquet')

schedule_e_2026= pd.read_parquet('../data/raw/schedule_e_2026.parquet')

vote_compare= pd.read_parquet('../data/raw/vote_compare.parquet')


In [3]:
bill_summaries_119.head(3).T

,0,1,2
actionDate,2025-01-03,2025-01-09,2025-01-06
actionDesc,Introduced in House,Introduced in House,Introduced in Senate
currentChamber,House,House,Senate
currentChamberCode,H,H,S
lastSummaryUpdateDate,2025-01-09T01:58:40Z,2025-01-13T12:49:11Z,2025-01-13T16:21:37Z
text,<p><strong>Illegitimate Court Counteraction Ac...,<p>This resolution disapproves of the Central ...,<p><strong>Laken Riley Act</strong></p><p>This...
updateDate,2025-01-09T01:58:53Z,2025-01-13T12:51:00Z,2025-01-13T16:21:43Z
versionCode,00,00,00
bill.congress,119,119,119
bill.number,23,27,5


In [4]:
sponsored_legislation.head(3).T

,0,1,2
introducedDate,2026-05-13,2026-07-22,2026-07-22
number,8800,NaN,NaN
title,National Defense Authorization Act for Fiscal ...,NaN,NaN
type,HR,NaN,NaN
url,https://api.congress.gov/v3/bill/119/hr/8800?f...,https://api.congress.gov/v3/amendment/119/hamd...,https://api.congress.gov/v3/amendment/119/hamd...
bioguide_id,R000575,R000575,R000575


In [5]:
from bs4 import BeautifulSoup
mysoup = BeautifulSoup(bill_summaries_119['text'][0], 'html.parser')
mysoup.text

'Illegitimate Court Counteraction ActThis bill imposes sanctions against foreign persons (individuals and entities) who assist the International Criminal Court (ICC) in investigating, arresting, detaining, or prosecuting certain individuals.The bill categorizes as protected persons (1) any U.S.\xa0individual, U.S. entity, or person in the United States, unless the United States is a state party to the Rome Statute of the ICC and provides formal consent to ICC jurisdiction; and (2) any foreign person that is a citizen or lawful resident of a U.S. ally that is not a state party to the Rome Statute or has not consented to ICC jurisdiction.If the ICC attempts to investigate, arrest, detain or prosecute a protected person, the President must impose visa- and property-blocking sanctions against the foreign persons that engaged in or materially assisted in such actions, as well as against foreign persons owned by, controlled by, or acting on behalf of such foreign persons. The President must 

In [6]:
sponsored_legislation[sponsored_legislation['url'].duplicated(keep=False)]

,introducedDate,number,title,type,url,bioguide_id
13937,2025-04-04,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,S001150
13938,2025-04-04,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,S001150
15083,2026-02-12,3872,A bill to amend title XVIII of the Social Secu...,S,https://api.congress.gov/v3/bill/119/s/3872?fo...,B001277
15085,2026-02-12,3872,A bill to amend title XVIII of the Social Secu...,S,https://api.congress.gov/v3/bill/119/s/3872?fo...,B001277
20372,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20373,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20374,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20375,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20379,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608
20380,2025-02-20,NaN,NaN,NaN,https://api.congress.gov/v3/amendment/119/samd...,R000608


In [7]:
sponsored_legislation = sponsored_legislation.drop_duplicates()
sponsored_legislation = sponsored_legislation[['bioguide_id', 'url']]

bills = pd.merge(bill_summaries_119, sponsored_legislation, left_on = ['bill.url'], right_on = ['url'], 
                 how = "outer", 
                 indicator = "matched", 
                 validate= "many_to_one")



In [8]:
bills['matched'].value_counts()

matched
right_only    20597
both           6146
left_only         0
Name: count, dtype: int64

In [9]:
bills[bills['matched']=="right_only"]['url']

0        https://api.congress.gov/v3/amendment/119/hamd...
1        https://api.congress.gov/v3/amendment/119/hamd...
2        https://api.congress.gov/v3/amendment/119/hamd...
3        https://api.congress.gov/v3/amendment/119/hamd...
4        https://api.congress.gov/v3/amendment/119/hamd...
                               ...                        
26733    https://api.congress.gov/v3/bill/119/sres/947?...
26734    https://api.congress.gov/v3/bill/119/sres/948?...
26735    https://api.congress.gov/v3/bill/119/sres/949?...
26736    https://api.congress.gov/v3/bill/119/sres/94?f...
26739    https://api.congress.gov/v3/bill/119/sres/97?f...
Name: url, Length: 20597, dtype: str

In [10]:
bills[bills['matched']=="both"]

,actionDate,actionDesc,currentChamber,currentChamberCode,lastSummaryUpdateDate,text,updateDate,versionCode,bill.congress,bill.number,bill.originChamber,bill.originChamberCode,bill.title,bill.type,bill.updateDateIncludingText,bill.url,bioguide_id,url,matched
7021,2026-05-13,Introduced in House,House,H,2026-06-24T14:07:00Z,<p>This concurrent resolution directs the Pres...,2026-06-24T14:07:20Z,00,119.0,100,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,F000477,https://api.congress.gov/v3/bill/119/hconres/1...,both
7022,2026-05-14,Introduced in House,House,H,2026-07-18T16:28:55Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:00Z,00,119.0,101,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,J000305,https://api.congress.gov/v3/bill/119/hconres/1...,both
7023,2026-05-15,Introduced in House,House,H,2026-07-17T20:36:05Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:10Z,00,119.0,102,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,R000621,https://api.congress.gov/v3/bill/119/hconres/1...,both
7024,2026-05-20,Introduced in House,House,H,2026-07-18T16:29:53Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:58Z,00,119.0,103,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-22,https://api.congress.gov/v3/bill/119/hconres/1...,A000380,https://api.congress.gov/v3/bill/119/hconres/1...,both
7025,2026-05-20,Introduced in House,House,H,2026-07-17T20:36:47Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:52Z,00,119.0,104,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-07-17,https://api.congress.gov/v3/bill/119/hconres/1...,B001313,https://api.congress.gov/v3/bill/119/hconres/1...,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26737,2025-02-25,Introduced in Senate,Senate,S,2026-09-23T16:14:06Z,<p>This resolution designates February 23-Marc...,2026-09-23T16:14:34Z,00,119.0,95,Senate,S,A resolution expressing support for the design...,SRES,2026-09-23,https://api.congress.gov/v3/bill/119/sres/95?f...,S001217,https://api.congress.gov/v3/bill/119/sres/95?f...,both
26738,2025-02-25,Introduced in Senate,Senate,S,2025-05-07T19:53:39Z,<p>This resolution designates February 24-Febr...,2025-05-07T19:54:03Z,00,119.0,96,Senate,S,A resolution designating the week of February ...,SRES,2025-07-01,https://api.congress.gov/v3/bill/119/sres/96?f...,C001035,https://api.congress.gov/v3/bill/119/sres/96?f...,both
26740,2025-02-26,Introduced in Senate,Senate,S,2025-05-08T19:34:05Z,<p>This resolution condemns China's Hong Kong ...,2025-05-08T19:34:25Z,00,119.0,98,Senate,S,A resolution condemning Beijing's destruction ...,SRES,2025-05-08,https://api.congress.gov/v3/bill/119/sres/98?f...,R000584,https://api.congress.gov/v3/bill/119/sres/98?f...,both
26741,2025-02-26,Introduced in Senate,Senate,S,2025-09-19T20:56:40Z,<p>This resolution recognizes Black History Mo...,2025-09-19T20:57:04Z,00,119.0,99,Senate,S,A resolution celebrating Black History Month.,SRES,2026-03-03,https://api.congress.gov/v3/bill/119/sres/99?f...,B001288,https://api.congress.gov/v3/bill/119/sres/99?f...,both


In [11]:
bills = bills.query("matched == 'both'")
bills = bills.drop('matched', axis=1)
bills

,actionDate,actionDesc,currentChamber,currentChamberCode,lastSummaryUpdateDate,text,updateDate,versionCode,bill.congress,bill.number,bill.originChamber,bill.originChamberCode,bill.title,bill.type,bill.updateDateIncludingText,bill.url,bioguide_id,url
7021,2026-05-13,Introduced in House,House,H,2026-06-24T14:07:00Z,<p>This concurrent resolution directs the Pres...,2026-06-24T14:07:20Z,00,119.0,100,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,F000477,https://api.congress.gov/v3/bill/119/hconres/1...
7022,2026-05-14,Introduced in House,House,H,2026-07-18T16:28:55Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:00Z,00,119.0,101,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,J000305,https://api.congress.gov/v3/bill/119/hconres/1...
7023,2026-05-15,Introduced in House,House,H,2026-07-17T20:36:05Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:10Z,00,119.0,102,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,R000621,https://api.congress.gov/v3/bill/119/hconres/1...
7024,2026-05-20,Introduced in House,House,H,2026-07-18T16:29:53Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:58Z,00,119.0,103,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-22,https://api.congress.gov/v3/bill/119/hconres/1...,A000380,https://api.congress.gov/v3/bill/119/hconres/1...
7025,2026-05-20,Introduced in House,House,H,2026-07-17T20:36:47Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:52Z,00,119.0,104,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-07-17,https://api.congress.gov/v3/bill/119/hconres/1...,B001313,https://api.congress.gov/v3/bill/119/hconres/1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26737,2025-02-25,Introduced in Senate,Senate,S,2026-09-23T16:14:06Z,<p>This resolution designates February 23-Marc...,2026-09-23T16:14:34Z,00,119.0,95,Senate,S,A resolution expressing support for the design...,SRES,2026-09-23,https://api.congress.gov/v3/bill/119/sres/95?f...,S001217,https://api.congress.gov/v3/bill/119/sres/95?f...
26738,2025-02-25,Introduced in Senate,Senate,S,2025-05-07T19:53:39Z,<p>This resolution designates February 24-Febr...,2025-05-07T19:54:03Z,00,119.0,96,Senate,S,A resolution designating the week of February ...,SRES,2025-07-01,https://api.congress.gov/v3/bill/119/sres/96?f...,C001035,https://api.congress.gov/v3/bill/119/sres/96?f...
26740,2025-02-26,Introduced in Senate,Senate,S,2025-05-08T19:34:05Z,<p>This resolution condemns China's Hong Kong ...,2025-05-08T19:34:25Z,00,119.0,98,Senate,S,A resolution condemning Beijing's destruction ...,SRES,2025-05-08,https://api.congress.gov/v3/bill/119/sres/98?f...,R000584,https://api.congress.gov/v3/bill/119/sres/98?f...
26741,2025-02-26,Introduced in Senate,Senate,S,2025-09-19T20:56:40Z,<p>This resolution recognizes Black History Mo...,2025-09-19T20:57:04Z,00,119.0,99,Senate,S,A resolution celebrating Black History Month.,SRES,2026-03-03,https://api.congress.gov/v3/bill/119/sres/99?f...,B001288,https://api.congress.gov/v3/bill/119/sres/99?f...


In [12]:
bills.columns = [x.lower().replace('.', '_') for x in bills.columns]
bills 

,actiondate,actiondesc,currentchamber,currentchambercode,lastsummaryupdatedate,text,updatedate,versioncode,bill_congress,bill_number,bill_originchamber,bill_originchambercode,bill_title,bill_type,bill_updatedateincludingtext,bill_url,bioguide_id,url
7021,2026-05-13,Introduced in House,House,H,2026-06-24T14:07:00Z,<p>This concurrent resolution directs the Pres...,2026-06-24T14:07:20Z,00,119.0,100,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,F000477,https://api.congress.gov/v3/bill/119/hconres/1...
7022,2026-05-14,Introduced in House,House,H,2026-07-18T16:28:55Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:00Z,00,119.0,101,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,J000305,https://api.congress.gov/v3/bill/119/hconres/1...
7023,2026-05-15,Introduced in House,House,H,2026-07-17T20:36:05Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:10Z,00,119.0,102,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,R000621,https://api.congress.gov/v3/bill/119/hconres/1...
7024,2026-05-20,Introduced in House,House,H,2026-07-18T16:29:53Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:58Z,00,119.0,103,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-22,https://api.congress.gov/v3/bill/119/hconres/1...,A000380,https://api.congress.gov/v3/bill/119/hconres/1...
7025,2026-05-20,Introduced in House,House,H,2026-07-17T20:36:47Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:52Z,00,119.0,104,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-07-17,https://api.congress.gov/v3/bill/119/hconres/1...,B001313,https://api.congress.gov/v3/bill/119/hconres/1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26737,2025-02-25,Introduced in Senate,Senate,S,2026-09-23T16:14:06Z,<p>This resolution designates February 23-Marc...,2026-09-23T16:14:34Z,00,119.0,95,Senate,S,A resolution expressing support for the design...,SRES,2026-09-23,https://api.congress.gov/v3/bill/119/sres/95?f...,S001217,https://api.congress.gov/v3/bill/119/sres/95?f...
26738,2025-02-25,Introduced in Senate,Senate,S,2025-05-07T19:53:39Z,<p>This resolution designates February 24-Febr...,2025-05-07T19:54:03Z,00,119.0,96,Senate,S,A resolution designating the week of February ...,SRES,2025-07-01,https://api.congress.gov/v3/bill/119/sres/96?f...,C001035,https://api.congress.gov/v3/bill/119/sres/96?f...
26740,2025-02-26,Introduced in Senate,Senate,S,2025-05-08T19:34:05Z,<p>This resolution condemns China's Hong Kong ...,2025-05-08T19:34:25Z,00,119.0,98,Senate,S,A resolution condemning Beijing's destruction ...,SRES,2025-05-08,https://api.congress.gov/v3/bill/119/sres/98?f...,R000584,https://api.congress.gov/v3/bill/119/sres/98?f...
26741,2025-02-26,Introduced in Senate,Senate,S,2025-09-19T20:56:40Z,<p>This resolution recognizes Black History Mo...,2025-09-19T20:57:04Z,00,119.0,99,Senate,S,A resolution celebrating Black History Month.,SRES,2026-03-03,https://api.congress.gov/v3/bill/119/sres/99?f...,B001288,https://api.congress.gov/v3/bill/119/sres/99?f...


In [13]:
bills.duplicated(['bill_url', 'versioncode']).value_counts()

False    6146
Name: count, dtype: int64

In [14]:
bills = bills.drop('url', axis=1)
bills.to_csv("../data/processed/bills.csv", index=False)

In [15]:
def pandas_df_to_dbml(df: pd.DataFrame, table_name: str) -> str:
    """
    Converts a pandas DataFrame to a DBML string.
    Args:
        df: The pandas DataFrame to convert.
        table_name: The name of the table in the DBML schema.
    Returns:
        A DBML string representing the DataFrame schema.
    """
    dbml_string = f"Table {table_name} {{\n"
    for column_name, column_type in df.dtypes.items():
        dbml_type = map_pandas_dtype_to_dbml_type(column_type)
        dbml_string += f"  {column_name} {dbml_type}\n"
    dbml_string += "}\n"
    return dbml_string
def map_pandas_dtype_to_dbml_type(dtype) -> str:
    """Maps a pandas dtype to a DBML type."""
    dtype_name = str(dtype)
    if "int" in dtype_name:
      return "int"
    if "float" in dtype_name:
      return "float"
    if "datetime" in dtype_name:
        return "datetime"
    return "varchar"

In [16]:
print(pandas_df_to_dbml(bills, 'bills'))

Table bills {
  actiondate varchar
  actiondesc varchar
  currentchamber varchar
  currentchambercode varchar
  lastsummaryupdatedate varchar
  text varchar
  updatedate varchar
  versioncode varchar
  bill_congress float
  bill_number varchar
  bill_originchamber varchar
  bill_originchambercode varchar
  bill_title varchar
  bill_type varchar
  bill_updatedateincludingtext varchar
  bill_url varchar
  bioguide_id varchar
}



### FEC schedule E

In [17]:
pd.set_option('display.max_rows', 128)
schedule_e_2026.info(verbose=True, max_cols=float('inf'))

<class 'pandas.DataFrame'>
RangeIndex: 15772 entries, 0 to 15771
Data columns (total 128 columns):
 #    Column                                   Non-Null Count  Dtype  
---   ------                                   --------------  -----  
 0    fec_id                                   15772 non-null  str    
 1    bioguide_id                              15772 non-null  str    
 2    action_code                              15772 non-null  str    
 3    action_code_full                         15772 non-null  str    
 4    amendment_indicator                      15772 non-null  str    
 5    amendment_number                         8031 non-null   float64
 6    back_reference_schedule_name             26 non-null     str    
 7    back_reference_transaction_id            26 non-null     str    
 8    candidate_first_name                     15772 non-null  str    
 9    candidate_id                             15772 non-null  str    
 10   candidate_last_name                      15

In [18]:
pd.set_option('display.max_rows', 128)
schedule_e_2026.head(3).T

,0,1,2
fec_id,H4AL06098,H4AL06098,H8AL02171
bioguide_id,P000609,P000609,M001212
action_code,A,A,A
action_code_full,ADD,ADD,ADD
amendment_indicator,N,N,N
amendment_number,0.0,NaN,0.0
back_reference_schedule_name,NaN,NaN,NaN
back_reference_transaction_id,NaN,NaN,NaN
candidate_first_name,GARY,GARY,FELIX
candidate_id,H4AL06098,H4AL06098,H8AL02171


In [19]:
expenditures = schedule_e_2026[['fec_id', 'transaction_id', 'expenditure_amount', 'committee.name', 'committee.committee_type_full', 'support_oppose_indicator']]

In [20]:
expenditures.duplicated(['transaction_id']).value_counts()

False    10284
True      5488
Name: count, dtype: int64

In [21]:
expenditures[expenditures.duplicated(['transaction_id'], keep=False)]

,fec_id,transaction_id,expenditure_amount,committee.name,committee.committee_type_full,support_oppose_indicator
2,H8AL02171,E0E0A659AE9C94E6CA52,15000.00,CONSTITUTIONAL CONSERVATIVES FUND,Super PAC (Independent Expenditure-Only),S
3,H8AL02171,E0E0A659AE9C94E6CA52,15000.00,CONSTITUTIONAL CONSERVATIVES FUND,Super PAC (Independent Expenditure-Only),S
4,H8AL02171,E77F981314DDE44B285A,109.20,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
5,H8AL02171,EC1FFD4F8FD274B84814,59.85,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
6,H8AL02171,E54E5B85464C6478AB6F,167.33,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
...,...,...,...,...,...,...
15767,S0WY00137,E19CD48FA103A4B08818,53.77,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
15768,S0WY00137,E85960C39314B40C189C,103.42,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
15769,S0WY00137,EFF4917D0897E4FB193F,25.42,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S
15770,S0WY00137,E84341A51E0FF43EE934,26.75,SENATE CONSERVATIVES FUND,Hybrid PAC (with Non-Contribution Account) - Q...,S


In [25]:
expenditures = expenditures.rename({'committee.name': 'committee_name',
                                    'committee.committee_type_full': 'committee_type_full'}, axis = 1)


In [26]:
print(pandas_df_to_dbml(expenditures, "expenditures"))

Table expenditures {
  fec_id varchar
  transaction_id varchar
  expenditure_amount float
  committee_name varchar
  committee_type_full varchar
  support_oppose_indicator varchar
}



In [28]:
expenditures.to_csv("../data/processed/expenditures.csv", index=False)